## Original - Number of images per class - Before Split

In [1]:
import os

dataset_path = "/home/i_am_helium/skin_clean_R_G_IR_aligned_1_hairless"
classes = ['bcc', 'bkl', 'mel', 'nevus']

# Check actual filenames first
for cls in classes:
    class_path = os.path.join(dataset_path, cls)
    files = os.listdir(class_path)
    print(f"\n{cls} - First 5 files:")
    for f in files[:5]:
        print(f"  {f}")
    break  # Just check first class to see pattern


bcc - First 5 files:
  2_943_15_in_7714___2022-06-29_10-37-26
  2_899_9_in_7543___2022-04-14_10-02-22
  2_385_37_in_4876_4877___2020-07-24_11-46-51
  2_964_10_in_7834___2022-08-08_10-07-50
  2_682_1_in_7043___2021-11-15_11-28-16


In [2]:
import os

dataset_path = "/home/i_am_helium/skin_clean_R_G_IR_aligned_1_hairless"
classes = ['bcc', 'bkl', 'mel', 'nevus']

print("=" * 70)
print(f"{'Class':<10} {'Label':<10} {'R':<10} {'G':<10} {'B':<10} {'Total R+G+B':<15}")
print("=" * 70)

totals = {'R': 0, 'G': 0, 'B': 0}

for idx, cls in enumerate(classes):
    class_path = os.path.join(dataset_path, cls)
    
    # Count subdirectories (each contains R.png, G.png, IR.png)
    subdirs = [d for d in os.listdir(class_path) 
               if os.path.isdir(os.path.join(class_path, d))]
    
    num_samples = len(subdirs)
    
    # Each sample has R, G, IR (B)
    r_count = num_samples
    g_count = num_samples
    b_count = num_samples
    
    total = r_count + g_count + b_count
    
    totals['R'] += r_count
    totals['G'] += g_count
    totals['B'] += b_count
    
    print(f"{cls:<10} {idx:<10} {r_count:<10} {g_count:<10} {b_count:<10} {total:<15}")

print("=" * 70)
total_all = totals['R'] + totals['G'] + totals['B']
print(f"{'TOTAL':<10} {'':<10} {totals['R']:<10} {totals['G']:<10} {totals['B']:<10} {total_all:<15}")

Class      Label      R          G          B          Total R+G+B    
bcc        0          449        449        449        1347           
bkl        1          250        250        250        750            
mel        2          537        537        537        1611           
nevus      3          526        526        526        1578           
TOTAL                 1762       1762       1762       5286           


`bkl` is the minority class

# After splitting

# Split the dataset - Five Fold Split

In [3]:
import sys
sys.path.insert(0, '/home/i_am_helium/my_libs')


In [4]:
import os
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# Build DataFrame
dataset_path = "/home/i_am_helium/skin_clean_R_G_IR_aligned_1_hairless"
classes = ['bcc', 'bkl', 'mel', 'nevus']

data = []
for label_idx, cls in enumerate(classes):
    class_path = os.path.join(dataset_path, cls)
    files = sorted(os.listdir(class_path)) 
    
    lesion_folders = sorted([d for d in os.listdir(class_path) 
                             if os.path.isdir(os.path.join(class_path, d))])
    
    for lesion_id in lesion_folders:
        data.append({
            'lesion_id': lesion_id,
            'label': label_idx,
            'class_name': cls,
            'folder_path': os.path.join(class_path, lesion_id)
        })

skinDf = pd.DataFrame(data)
print(f"Total samples: {len(skinDf)}")
print(skinDf.groupby('class_name')['lesion_id'].count())

# Create folds
def create_stratified_group_folds(df, group_col='lesion_id', label_col='label', n_splits=5):
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)

    X = df.index.values
    y = df[label_col].values
    groups = df[group_col].values

    folds = []
    for fold_idx, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups)):
        train_fold_df = df.iloc[train_idx].reset_index(drop=True).copy()
        val_fold_df = df.iloc[val_idx].reset_index(drop=True).copy()
        folds.append((fold_idx, train_fold_df, val_fold_df))

    return folds

folds = create_stratified_group_folds(skinDf, group_col='lesion_id', label_col='label', n_splits=5)

print(f"\nNumber of folds: {len(folds)}")
for fold_idx, train_fold_df, val_fold_df in folds:
    print(f"\nFold {fold_idx+1}")
    print("Train rows:", len(train_fold_df), "| Train lesions:", train_fold_df['lesion_id'].nunique())
    print("Val rows  :", len(val_fold_df), "| Val lesions  :", val_fold_df['lesion_id'].nunique())
    print("Train class dist:", train_fold_df['label'].value_counts().sort_index().to_dict())
    print("Val class dist  :", val_fold_df['label'].value_counts().sort_index().to_dict())

Total samples: 1762
class_name
bcc      449
bkl      250
mel      537
nevus    526
Name: lesion_id, dtype: int64

Number of folds: 5

Fold 1
Train rows: 1409 | Train lesions: 1409
Val rows  : 353 | Val lesions  : 353
Train class dist: {0: 354, 1: 200, 2: 428, 3: 427}
Val class dist  : {0: 95, 1: 50, 2: 109, 3: 99}

Fold 2
Train rows: 1409 | Train lesions: 1409
Val rows  : 353 | Val lesions  : 353
Train class dist: {0: 361, 1: 200, 2: 428, 3: 420}
Val class dist  : {0: 88, 1: 50, 2: 109, 3: 106}

Fold 3
Train rows: 1410 | Train lesions: 1410
Val rows  : 352 | Val lesions  : 352
Train class dist: {0: 363, 1: 200, 2: 433, 3: 414}
Val class dist  : {0: 86, 1: 50, 2: 104, 3: 112}

Fold 4
Train rows: 1410 | Train lesions: 1410
Val rows  : 352 | Val lesions  : 352
Train class dist: {0: 360, 1: 200, 2: 427, 3: 423}
Val class dist  : {0: 89, 1: 50, 2: 110, 3: 103}

Fold 5
Train rows: 1410 | Train lesions: 1410
Val rows  : 352 | Val lesions  : 352
Train class dist: {0: 358, 1: 200, 2: 432, 3: 42

In [5]:
# Table showing the number of photos BEFORE augmentation, BY FOLD
label_map_display = {0: 'bcc', 1: 'bkl', 2: 'mel', 3: 'nevus'}

for fold_idx, train_fold_df, val_fold_df in folds:
    for split_name, split_df in [("TRAIN", train_fold_df), ("VAL", val_fold_df)]:

        print(f"\nFold {fold_idx+1} — {split_name}")
        print("=" * 65)
        print(f"{'Class':<8} {'Label':<8} {'R':<8} {'G':<8} {'B':<8} {'Total R+G+B':<14}")
        print("=" * 65)

        total_samples = len(split_df)
        missing = []

        for label, dx in label_map_display.items():
            class_df = split_df[split_df['label'] == label]
            count = len(class_df)
            count_R = count
            count_G = count
            count_B = count
            total_channels = count_R + count_G + count_B

            print(f"{dx:<8} {label:<8} {count_R:<8} {count_G:<8} {count_B:<8} {total_channels:<14}")

            if count == 0:
                missing.append(dx)

        print("=" * 65)
        total_channels_all = total_samples * 3

        print(f"{'TOTAL':<8} {'':<8} {total_samples:<8} {total_samples:<8} {total_samples:<8} {total_channels_all:<14}")

        print(f"Missing classes: {missing if missing else 'None'}")


Fold 1 — TRAIN
Class    Label    R        G        B        Total R+G+B   
bcc      0        354      354      354      1062          
bkl      1        200      200      200      600           
mel      2        428      428      428      1284          
nevus    3        427      427      427      1281          
TOTAL             1409     1409     1409     4227          
Missing classes: None

Fold 1 — VAL
Class    Label    R        G        B        Total R+G+B   
bcc      0        95       95       95       285           
bkl      1        50       50       50       150           
mel      2        109      109      109      327           
nevus    3        99       99       99       297           
TOTAL             353      353      353      1059          
Missing classes: None

Fold 2 — TRAIN
Class    Label    R        G        B        Total R+G+B   
bcc      0        361      361      361      1083          
bkl      1        200      200      200      600           
mel      2

In [6]:
# Check lesion overlap between train and val
for fold_idx, train_fold_df, val_fold_df in folds:
    train_lesions = set(train_fold_df['lesion_id'])
    val_lesions = set(val_fold_df['lesion_id'])

    overlap = train_lesions & val_lesions
    print(f"Fold {fold_idx+1} lesion overlap:", len(overlap))

Fold 1 lesion overlap: 0
Fold 2 lesion overlap: 0
Fold 3 lesion overlap: 0
Fold 4 lesion overlap: 0
Fold 5 lesion overlap: 0


## To send to Image Preprocessing - fold

In [7]:
import pickle
with open('folds_5cv.pkl', 'wb') as f:
    pickle.dump(folds, f)
print("Saved folds to folds_5cv.pkl")

Saved folds to folds_5cv.pkl
